In [3]:
!pip install -U pip
!pip install -U torch==2.2.2 torchvision==0.17.2 scikit-learn==1.4.2 numpy==1.26.4 pandas==2.2.2 matplotlib==3.8.4

In [1]:
import torch
import torchvision

print(torch.__version__)
print(torchvision.__version__)

2.10.0+cu128
0.25.0+cu128


In [2]:
!nvidia-smi

Thu Jun  4 18:35:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU: Tesla T4


In [6]:
!git clone https://github.com/Rohit-Kundu/Fuzzy-Integral-Covid-Detection.git
%cd Fuzzy-Integral-Covid-Detection


Cloning into 'Fuzzy-Integral-Covid-Detection'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 44 (delta 21), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 14.29 KiB | 4.76 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/kaggle/working/Fuzzy-Integral-Covid-Detection


In [6]:
import os, shutil, random #80-20 split
random.seed(42)

# Dataset is available as Kaggle input when you Add Data → sarscov2-ctscan-dataset
DATA_INPUT = "/kaggle/input/datasets/plameneduardo/sarscov2-ctscan-dataset"
DATA_DIR   = "/kaggle/working/data"

for split in ['train', 'val']:
    for cls in ['COVID', 'non-COVID']:
        os.makedirs(f"{DATA_DIR}/{split}/{cls}", exist_ok=True)

for cls in ['COVID', 'non-COVID']:
    files = sorted(os.listdir(f"{DATA_INPUT}/{cls}"))
    random.shuffle(files)
    n_train = int(0.8 * len(files))
    for i, f in enumerate(files):
        split = 'train' if i < n_train else 'val'
        shutil.copy(f"{DATA_INPUT}/{cls}/{f}", f"{DATA_DIR}/{split}/{cls}/{f}")

# Verify counts
for split in ['train', 'val']:
    for cls in ['COVID', 'non-COVID']:
        n = len(os.listdir(f"{DATA_DIR}/{split}/{cls}"))
        print(f"{split}/{cls}: {n}")

train/COVID: 1001
train/non-COVID: 983
val/COVID: 251
val/non-COVID: 246


In [8]:
#70-30 split
"""
split_data_7030.py  —  Corrected 70/30 stratified split
=========================================================
ROOT CAUSE DISCOVERED:
  The paper used a 70/30 train/val split, NOT 80/20.
  Proof: ensemble.py hardcodes labels as range(376) + range(369) = 745 val samples.
  With 1252 COVID + 1230 non-COVID (2482 total):
    int(0.70 * 1252) = 876 train  → 376 val   ✓ matches paper
    int(0.70 * 1230) = 861 train  → 369 val   ✓ matches paper
  An 80/20 split gives only 251+246 = 497 val samples (wrong).

USAGE (Kaggle):
  Run this cell BEFORE main.py. Adjust DATA_INPUT to your dataset path.
"""

import os
import shutil
import random

# ── Config ───────────────────────────────────────────────────────────────────
SEED       = 42
TRAIN_RATIO = 0.70          # 70% train, 30% val — matches the paper exactly
CLASSES    = ['COVID', 'non-COVID']

# Adjust these paths for your environment:
DATA_INPUT = "/kaggle/input/datasets/plameneduardo/sarscov2-ctscan-dataset"   # raw Kaggle dataset
DATA_DIR   = "/kaggle/working/data"                    # output split directory

# ── Build directory structure ────────────────────────────────────────────────
for split in ['train', 'val']:
    for cls in CLASSES:
        os.makedirs(f"{DATA_DIR}/{split}/{cls}", exist_ok=True)

# ── Split and copy files ─────────────────────────────────────────────────────
random.seed(SEED)

for cls in CLASSES:
    src_dir = os.path.join(DATA_INPUT, cls)
    all_files = sorted(os.listdir(src_dir))  # sort first for determinism
    random.shuffle(all_files)                # then shuffle with fixed seed

    n_train = int(TRAIN_RATIO * len(all_files))   # floor division — same as paper

    for i, fname in enumerate(all_files):
        split  = 'train' if i < n_train else 'val'
        dst    = os.path.join(DATA_DIR, split, cls, fname)
        shutil.copy(os.path.join(src_dir, fname), dst)

# ── Verify counts ────────────────────────────────────────────────────────────
print("=== Split Verification ===")
total_val = 0
for split in ['train', 'val']:
    for cls in CLASSES:
        n = len(os.listdir(f"{DATA_DIR}/{split}/{cls}"))
        total_val += n if split == 'val' else 0
        status = ""
        if split == 'val' and cls == 'COVID':
            status = "✓" if n == 376 else f"✗ expected 376"
        if split == 'val' and cls == 'non-COVID':
            status = "✓" if n == 369 else f"✗ expected 369"
        print(f"  {split}/{cls}: {n} {status}")

print(f"\n  Val total: {total_val}  {'✓ matches paper (745)' if total_val == 745 else '✗ expected 745'}")
print(f"\n  Ready to run: python main.py --data_directory {DATA_DIR} --epochs 100")

=== Split Verification ===
  train/COVID: 876 
  train/non-COVID: 860 
  val/COVID: 376 ✓
  val/non-COVID: 369 ✓

  Val total: 745  ✓ matches paper (745)

  Ready to run: python main.py --data_directory /kaggle/working/data --epochs 100


In [18]:
!python main.py --data_directory /kaggle/working/data --epochs 100  #70-30 

Using device: cuda:0
Classes: ['COVID', 'non-COVID']
Train: 1736 | Val: 745
Val class counts: {'COVID': 376, 'non-COVID': 369}
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG11_BN_Weights.IMAGENET1K_V1`. You can also use `weights=VGG11_BN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg11_bn-6002323d.pth" to /root/.cache/torch/hub/checkpoints/vgg11_bn-6002323d.pth
100%|█████████████████████████████████████████| 507M/507M [00:02<00:00, 227MB/s]
Epoch 1/100
----------
train Loss: 0

In [22]:
!python main.py --data_directory /kaggle/working/data --epochs 100 #80-20 split results

Using device: cuda:0
Classes: ['COVID', 'non-COVID']
Train: 1984 | Val: 497
Val class counts: {'COVID': 251, 'non-COVID': 246}
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG11_BN_Weights.IMAGENET1K_V1`. You can also use `weights=VGG11_BN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Epoch 1/100
----------
train Loss: 0.3683 Acc: 0.8201
val Loss: 0.1872 Acc: 0.9256
==> Model Saved

Epoch 2/100
----------
train Loss: 0.1065 Acc: 0.9556
val Loss: 0.1387 Acc: 0.9356
==> Model Saved

Epoch 3/100
----------
train Loss: 0.0271 Acc: 0.9

In [17]:
!cp "/kaggle/input/models/ibadatali/updated-sugeno-and-ensemble/pytorch/default/1/sugeno_integral.py" "/kaggle/working/Fuzzy-Integral-Covid-Detection"

In [13]:
# rm -rf /kaggle/working/Fuzzy-Integral-Covid-Detection/probability_extraction.py

In [3]:
%cd /kaggle/working
!zip -r Fuzzy-Integral-Covid-Detection.zip

/kaggle/working

zip error: Nothing to do! (Fuzzy-Integral-Covid-Detection.zip)
